In [28]:
import ee
import pandas as pd

EE_PROJECT=''

CSV_PATH = r"D:POI records data\china_datacenter_POI.csv"
df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")


OUTPUT_PATH = CSV_PATH.replace(".csv", "_with_ERA5_SRTM.csv")

START = '2024-01-01'
END = '2024-12-31'
# ===========================

try:
    ee.Initialize(project=EE_PROJECT)
   


df = pd.read_csv(CSV_PATH, encoding="utf-8-sig")


assert 'longitude' in df.columns and 'latitude' in df.columns


features = []
for idx, row in df.iterrows():
    pt = ee.Geometry.Point([float(row['longitude']), float(row['latitude'])])
    feat = ee.Feature(pt, {"id": idx})
    features.append(feat)

fc_points = ee.FeatureCollection(features)


era = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR") \
    .filterDate(START, END).mean()


temp_band = "temperature_2m" if "temperature_2m" in era.bandNames().getInfo() else None



prec_band = "total_precipitation_sum" if "total_precipitation_sum" in era.bandNames().getInfo() else None


era = era.select([temp_band, prec_band])


srtm = ee.Image("USGS/SRTMGL1_003").select("elevation")


temp_c = era.select(temp_band).subtract(273.15).rename("temp_C")
prec_mm = era.select(prec_band).multiply(1000).rename("precip_mm")
elev = srtm.rename("elevation_m")

full_img = temp_c.addBands(prec_mm).addBands(elev)



print("Start")
result_fc = full_img.reduceRegions(
    collection=fc_points,
    reducer=ee.Reducer.first(),
    scale=10000  
)


result = result_fc.getInfo()['features']




rows = []
for f in result:
    props = f['properties']
    idx = props.get("id")
    temp = props.get("temp_C")
    prec = props.get("precip_mm")
    elev = props.get("elevation_m")

    rows.append({
        "id": idx,
        "temp_C": temp,
        "precip_mm": prec,
        "elevation_m": elev
    })

df_res = pd.DataFrame(rows).sort_values("id").reset_index(drop=True)


df_out = pd.concat([df, df_res.drop(columns=['id'])], axis=1)


df_out.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("✔ 已写入：", OUTPUT_PATH)



EE 初始化成功
成功创建 FeatureCollection，共 1005 个点
开始 reduceRegions 批量抽样 ……（云端执行，无需等待太久）
reduceRegions 完成，开始转换为 DataFrame
✔ 已写入： D:\杨紫薇\数据论文\A 10-m resolution spatial distribution map of data centers in major provinces of China in 2024\POI records data\china_datacenter_POI1_with_ERA5_SRTM.csv
